# Checkpoint-1200 → Full-24 Order / First / Last Direct Training + Pairwise Agreement Metrics

시작 어댑터:

```text
/content/drive/MyDrive/SNU_AI_Challenge/
qwen2vl_7b_multitask_bipair_conditional_v1/
runs/20260721_011312/
multitask_bipair_conditional/checkpoint-1200
```

이 노트북은 다음 실험을 수행합니다.

1. **24개 모든 순열**을 정확히 점수화합니다.
2. 독립 재계산 방식과 **공통 prefix KV-cache 방식**의 점수·확률·순위·주변확률이 같은지 먼저 검사합니다.
3. 전체 순열 분포에 `order CE + order Brier`를 적용합니다.
4. 같은 24개 분포에서 만든 `first marginal`, `last marginal`에 각각 `CE + Brier`를 직접 적용합니다.
5. 매 checkpoint에서 전체/첫/마지막의 **확률, margin, entropy, normalized entropy, Brier, AUC**를 CSV로 남깁니다.
6. 전체 normalized entropy 분포의 largest-gap으로 `global_low_confidence` 집단도 자동 표시합니다.
7. 이미 계산한 24개 순열 확률에서 `P(i before j)`를 합산하여 **pairwise earliest/latest 예측**을 만들고, marginal first/last 및 conditional endpoint 예측과의 불일치·합의 패턴을 추가로 저장합니다.

## Pairwise 지표의 계산 비용

추가 pairwise 지표는 별도의 모델 호출이나 별도 pairwise 프롬프트를 사용하지 않습니다.
이미 계산된 full-24 확률에서 다음 값을 합산해 파생합니다.

```text
P(i before j) = sum P(order), for every order where i appears before j
```

따라서 **학습 objective, gradient, 모델 forward 횟수는 기존 노트북과 동일**하며 평가 CSV와 summary 열만 추가됩니다.

## 정확한 gradient를 유지하는 방법

KV-cache는 전체 24개 점수를 빠르게 계산하는 데 사용합니다. 학습 시에는 이 점수 벡터에서
`dL/dscore_i`를 정확히 구한 뒤, 후보를 작은 batch로 다시 점수화하여

```text
sum_i (dL/dscore_i) * score_i
```

를 역전파합니다. 따라서 mutable KV-cache 객체에 직접 backward하지 않으면서도, **full-24 loss의 chain-rule gradient는 정확히 유지**합니다. Dropout은 0으로 고정하여 두 점수화 과정이 같은 정책을 사용하게 합니다.


In [ ]:

# 1) Install dependencies, then restart runtime once.
# After restart, run this cell again and continue.
import os
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")
import subprocess
import sys
from pathlib import Path

MARKER = Path("/content/.snu_full24_multilevel_deps_installed")

if Path("/content").exists() and not MARKER.exists():
    packages = [
        "transformers>=4.49.0,<4.54.0",
        "accelerate>=0.34.0",
        "bitsandbytes>=0.46.1",
        "peft",
        "qwen-vl-utils",
        "huggingface_hub",
        "hf_xet",
        "modelscope",
        "pandas==2.2.2",
        "safetensors>=0.4.5",
    ]
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-U", *packages])
    MARKER.write_text("ok")
    print("Dependencies installed. Restarting runtime. Run this cell again after restart.")
    os.kill(os.getpid(), 9)
elif Path("/content").exists():
    print("Dependencies already installed. Continue.")
else:
    print("Local environment detected. Skipping Colab dependency install.")


In [ ]:

# 2) Setup: Drive, data, model cache, paths, and experiment configuration
from google.colab import drive
from pathlib import Path
import os
import shutil

# Colab sometimes has a local /content/drive directory before mounting.
drive_root = Path("/content/drive")
if drive_root.exists() and not os.path.ismount(str(drive_root)) and any(drive_root.iterdir()):
    print("Removing local pre-mount /content/drive contents:", sorted(str(p) for p in drive_root.iterdir())[:20])
    shutil.rmtree(drive_root)
drive_root.mkdir(parents=True, exist_ok=True)
drive.mount("/content/drive")

import ast
import copy
import gc
import glob
import hashlib
import itertools
import json
import math
import random
import re
import subprocess
import zipfile
from datetime import datetime

os.environ["HF_HUB_DISABLE_XET"] = "1"
os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "0"

import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from PIL import Image
from tqdm.auto import tqdm
from transformers import (
    AutoModelForVision2Seq,
    AutoProcessor,
    BitsAndBytesConfig,
    set_seed,
)
try:
    from transformers import Qwen2VLForConditionalGeneration
except ImportError:
    Qwen2VLForConditionalGeneration = AutoModelForVision2Seq
from peft import PeftModel, prepare_model_for_kbit_training
import bitsandbytes as bnb
from transformers.utils import logging as transformers_logging

transformers_logging.set_verbosity_error()

SNU_ROOT = Path("/content/drive/MyDrive/SNU_AI_Challenge")
assert SNU_ROOT.exists(), SNU_ROOT

ZIP_PATH = SNU_ROOT / "snuaichallenge.zip"
DATA_DIR = Path("/content/snuaichallenge_data")
TRAIN_CSV = DATA_DIR / "train.csv"
TEST_CSV = DATA_DIR / "test.csv"
TRAIN_IMAGE_DIR = DATA_DIR / "train"
TEST_IMAGE_DIR = DATA_DIR / "test"

if not TRAIN_CSV.exists() or not TRAIN_IMAGE_DIR.is_dir():
    print("Extracting:", ZIP_PATH)
    with zipfile.ZipFile(ZIP_PATH) as zip_file:
        zip_file.extractall("/content/")

assert TRAIN_CSV.exists(), TRAIN_CSV
assert TEST_CSV.exists(), TEST_CSV
assert TRAIN_IMAGE_DIR.is_dir(), TRAIN_IMAGE_DIR
assert TEST_IMAGE_DIR.is_dir(), TEST_IMAGE_DIR

MODEL_REPO_ID = "Qwen/Qwen2-VL-7B-Instruct"
USE_MODELSCOPE_BASE_MODEL = False
DRIVE_MODEL_DIR = SNU_ROOT / "model_cache/Qwen2-VL-7B-Instruct"
LOCAL_MODEL_DIR = Path("/content/Qwen2-VL-7B-Instruct")


def print_runtime_storage():
    print("Storage check for /content:")
    try:
        subprocess.run(["df", "-h", "/content"], check=False)
    except Exception as exc:
        total, used, free = shutil.disk_usage("/content")
        print(f"/content free: {free / (1024 ** 3):.1f} GB / total: {total / (1024 ** 3):.1f} GB ({exc})")
    if torch.cuda.is_available():
        free, total = torch.cuda.mem_get_info()
        print(f"GPU memory free: {free / (1024 ** 3):.1f} GB / total: {total / (1024 ** 3):.1f} GB")


def model_cache_is_complete(model_dir):
    model_dir = Path(model_dir)
    if not (model_dir / "config.json").exists():
        return False
    has_weight = (model_dir / "model.safetensors.index.json").exists() or bool(list(model_dir.glob("*.safetensors")))
    has_processor = any((model_dir / name).exists() for name in [
        "preprocessor_config.json", "processor_config.json", "tokenizer.json", "tokenizer_config.json"
    ])
    return bool(has_weight and has_processor)


def copy_drive_cache_to_local():
    if not model_cache_is_complete(DRIVE_MODEL_DIR):
        raise FileNotFoundError(f"Drive model cache is incomplete or missing: {DRIVE_MODEL_DIR}")
    if model_cache_is_complete(LOCAL_MODEL_DIR):
        print("Using existing local model:", LOCAL_MODEL_DIR)
        return
    print("Copying base model from Drive to Colab local disk...")
    if LOCAL_MODEL_DIR.exists():
        shutil.rmtree(LOCAL_MODEL_DIR)
    shutil.copytree(DRIVE_MODEL_DIR, LOCAL_MODEL_DIR)
    assert model_cache_is_complete(LOCAL_MODEL_DIR), LOCAL_MODEL_DIR


def download_base_model_to_local_and_cache():
    print("Base model cache not found. Downloading:", MODEL_REPO_ID)
    if LOCAL_MODEL_DIR.exists() and not model_cache_is_complete(LOCAL_MODEL_DIR):
        shutil.rmtree(LOCAL_MODEL_DIR)
    from huggingface_hub import snapshot_download
    snapshot_download(
        repo_id=MODEL_REPO_ID,
        local_dir=str(LOCAL_MODEL_DIR),
        token=os.environ.get("HF_TOKEN"),
        max_workers=8,
    )
    assert model_cache_is_complete(LOCAL_MODEL_DIR), LOCAL_MODEL_DIR
    DRIVE_MODEL_DIR.parent.mkdir(parents=True, exist_ok=True)
    tmp = Path(str(DRIVE_MODEL_DIR) + ".tmp")
    if tmp.exists():
        shutil.rmtree(tmp)
    shutil.copytree(LOCAL_MODEL_DIR, tmp)
    if DRIVE_MODEL_DIR.exists():
        shutil.rmtree(DRIVE_MODEL_DIR)
    os.replace(tmp, DRIVE_MODEL_DIR)


def ensure_base_model_path():
    print_runtime_storage()
    if model_cache_is_complete(LOCAL_MODEL_DIR):
        print("Using local model:", LOCAL_MODEL_DIR)
    elif model_cache_is_complete(DRIVE_MODEL_DIR):
        copy_drive_cache_to_local()
    elif not USE_MODELSCOPE_BASE_MODEL:
        download_base_model_to_local_and_cache()
    else:
        from modelscope import snapshot_download as modelscope_snapshot_download
        model_dir = modelscope_snapshot_download(MODEL_REPO_ID, cache_dir="/content/modelscope_cache")
        if LOCAL_MODEL_DIR.exists():
            shutil.rmtree(LOCAL_MODEL_DIR)
        shutil.copytree(model_dir, LOCAL_MODEL_DIR)
    return str(LOCAL_MODEL_DIR)


MODEL_ID = ensure_base_model_path()
MODEL_LOCAL_FILES_ONLY = True

OUTPUT_ROOT = SNU_ROOT / "qwen2vl_7b_multitask_bipair_conditional_v1"
BASE_RUN_ID = "20260721_011312"
BASE_CHECKPOINT_NAME = "checkpoint-1200"
BASE_CHECKPOINT_DIR = (
    OUTPUT_ROOT / "runs" / BASE_RUN_ID /
    "multitask_bipair_conditional" / BASE_CHECKPOINT_NAME
)
assert BASE_CHECKPOINT_DIR.is_dir(), BASE_CHECKPOINT_DIR

# Stable ID: reconnecting to Colab resumes the same experiment.
RUN_ID = "20260723_full24_order_first_last_from_checkpoint1200_v2_pairwise_metrics"
RUN_ROOT = OUTPUT_ROOT / "runs" / RUN_ID
OUTPUT_DIR = RUN_ROOT / "full24_order_first_last_calibrated_pairwise_metrics"
CHECKPOINT_ROOT = OUTPUT_DIR / "checkpoints"
EVAL_DIR = OUTPUT_DIR / "eval"
QUICK_EVAL_DIR = EVAL_DIR / "quick20"
FULL_EVAL_DIR = EVAL_DIR / "matched100"
for path in [RUN_ROOT, OUTPUT_DIR, CHECKPOINT_ROOT, EVAL_DIR, QUICK_EVAL_DIR, FULL_EVAL_DIR]:
    path.mkdir(parents=True, exist_ok=True)

# Core reproducibility
SEED = 42
VALID_RATIO = 0.10
MATCHED_EVAL_SEED = SEED + 1
MATCHED_POOL_ROWS = 100
QUICK_EVAL_ROWS = 20
FINAL_EVAL_ROWS = 100

# Training
TRAIN_STEPS = 1000
LEARNING_RATE = 2e-7
MAX_GRAD_NORM = 1.0
SAVE_STEPS = 100
EVAL_STEPS = 200
LOGGING_STEPS = 10
AUTO_RESUME = True

# Full-24 objective. Set Brier weights to 0.0 for CE-only ablation.
LOSS_WEIGHTS = {
    "order_ce": 1.00,
    "first_ce": 0.20,
    "last_ce": 0.20,
    "order_brier": 0.02,
    "first_brier": 0.02,
    "last_brier": 0.02,
}
DISTRIBUTION_TEMPERATURE = 1.0

# T4-safe scoring
NO_GRAD_RECOMPUTE_BATCH_SIZE = 1
GRAD_RESCORE_BATCH_SIZE = 1
KV_DIAGNOSTIC_SAMPLES = 3
KV_SCORE_TOLERANCE = 2e-2
KV_PROB_TOLERANCE = 2e-3
RESCORE_SCORE_TOLERANCE = 3e-2
PREFER_KV_FOR_DISTRIBUTION = True

# Evaluation / output
RUN_INITIAL_QUICK_EVAL = True
RUN_INITIAL_MATCHED100_EVAL = True
RUN_FINAL_MATCHED100_EVAL = True
SAVE_ALL_24_PROBABILITIES = True

MIN_PIXELS = 128 * 28 * 28
MAX_PIXELS = 256 * 28 * 28
PERMUTATIONS = list(itertools.permutations([1, 2, 3, 4]))
PERMUTATION_TO_INDEX = {tuple(order): idx for idx, order in enumerate(PERMUTATIONS)}

set_seed(SEED)
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)


def checkpoint_step(path):
    match = re.search(r"full24-step-(\d+)$", Path(path).name)
    return int(match.group(1)) if match else -1


def find_latest_full24_checkpoint():
    candidates = [Path(p) for p in glob.glob(str(CHECKPOINT_ROOT / "checkpoint-1200-full24-step-*"))]
    candidates = [p for p in candidates if (p / "adapter_config.json").exists()]
    return max(candidates, key=checkpoint_step) if candidates else None


LATEST_CHECKPOINT = find_latest_full24_checkpoint() if AUTO_RESUME else None
START_ADAPTER_DIR = LATEST_CHECKPOINT or BASE_CHECKPOINT_DIR
START_STEP = checkpoint_step(LATEST_CHECKPOINT) if LATEST_CHECKPOINT else 0

run_config = {
    "experiment": "checkpoint1200_full24_order_first_last_calibrated_pairwise_metrics",
    "base_checkpoint_dir": str(BASE_CHECKPOINT_DIR),
    "start_adapter_dir": str(START_ADAPTER_DIR),
    "start_step": START_STEP,
    "output_dir": str(OUTPUT_DIR),
    "train_steps": TRAIN_STEPS,
    "learning_rate": LEARNING_RATE,
    "loss_weights": LOSS_WEIGHTS,
    "distribution_temperature": DISTRIBUTION_TEMPERATURE,
    "save_steps": SAVE_STEPS,
    "eval_steps": EVAL_STEPS,
    "seed": SEED,
}
with open(RUN_ROOT / "run_config.json", "w", encoding="utf-8") as handle:
    json.dump(run_config, handle, ensure_ascii=False, indent=2)

print("Base checkpoint:", BASE_CHECKPOINT_DIR)
print("Start adapter:", START_ADAPTER_DIR)
print("Start step:", START_STEP)
print("Output:", OUTPUT_DIR)
print("Loss weights:", LOSS_WEIGHTS)


In [ ]:

# 3) Data split, order examples, and fixed matched evaluation sets
def parse_answer(answer):
    result = answer if isinstance(answer, list) else ast.literal_eval(str(answer))
    result = [int(value) for value in result]
    if len(result) != 4 or sorted(result) != [1, 2, 3, 4]:
        raise ValueError(f"Invalid Answer: {answer}")
    return result


def order_to_sequence(answer):
    return [input_index + 1 for input_index, _ in sorted(enumerate(answer), key=lambda item: item[1])]


def compact_order(order):
    return " ".join(str(int(value)) for value in order)


def row_image_paths(row, image_root=TRAIN_IMAGE_DIR):
    sample_id = str(row["Id"])
    return [str(Path(image_root) / sample_id / str(row[f"Input_{i}"])) for i in range(1, 5)]


def load_rgb(path):
    with Image.open(path) as image:
        return image.convert("RGB").copy()


def row_to_order_example(row_index, row, image_root=TRAIN_IMAGE_DIR):
    answer = [int(value) for value in row["Answer_list"]]
    order = order_to_sequence(answer)
    return {
        "row_index": int(row_index),
        "sample_id": str(row["Id"]),
        "sentence": "" if pd.isna(row["Sentence"]) else str(row["Sentence"]),
        "answer": answer,
        "order": order,
        "task_type": "order",
        "target": compact_order(order),
        "image_paths": row_image_paths(row, image_root),
    }


def task_instruction(example):
    return (
        f"Caption:\n{example['sentence']}\n\n"
        "Order all four images from earliest to latest in the story.\n"
        "Answer only four frame numbers separated by spaces, for example: 1 2 3 4."
    )


def make_messages(example, include_answer=False):
    content = []
    for idx, _ in enumerate(example["image_paths"], start=1):
        content.append({"type": "text", "text": f"\nImage {idx}:"})
        content.append({"type": "image"})
    content.append({"type": "text", "text": "\n\n" + task_instruction(example)})
    messages = [{"role": "user", "content": content}]
    if include_answer:
        messages.append({"role": "assistant", "content": str(example["target"])})
    return messages


train_df = pd.read_csv(TRAIN_CSV)
test_df = pd.read_csv(TEST_CSV)
train_df["Id"] = train_df["Id"].astype(str)
test_df["Id"] = test_df["Id"].astype(str)
train_df["Answer_list"] = train_df["Answer"].apply(parse_answer)

split_root = SNU_ROOT / "id_splits" / "qwen2vl_lgt_order_refine_20260714_003635"
if (split_root / "train_ids.json").exists() and (split_root / "validation_ids.json").exists():
    train_ids = set(str(x) for x in json.load(open(split_root / "train_ids.json", "r", encoding="utf-8")))
    valid_ids = set(str(x) for x in json.load(open(split_root / "validation_ids.json", "r", encoding="utf-8")))
else:
    unique_ids = train_df["Id"].unique().copy()
    rng = np.random.default_rng(SEED)
    rng.shuffle(unique_ids)
    valid_size = max(1, int(len(unique_ids) * VALID_RATIO))
    valid_ids = set(unique_ids[:valid_size])
    train_ids = set(unique_ids[valid_size:])

training_df = train_df[train_df["Id"].isin(train_ids)].reset_index(drop=True)
validation_df = train_df[train_df["Id"].isin(valid_ids)].reset_index(drop=True)

matched_pool_df = validation_df.sample(
    n=min(MATCHED_POOL_ROWS, len(validation_df)),
    random_state=MATCHED_EVAL_SEED,
).reset_index(drop=True)
quick_eval_df = matched_pool_df.iloc[:min(QUICK_EVAL_ROWS, len(matched_pool_df))].copy().reset_index(drop=True)
full_eval_df = matched_pool_df.iloc[:min(FINAL_EVAL_ROWS, len(matched_pool_df))].copy().reset_index(drop=True)

quick_ids = quick_eval_df["Id"].astype(str).tolist()
full_ids = full_eval_df["Id"].astype(str).tolist()
quick_hash = hashlib.sha1("\n".join(quick_ids).encode("utf-8")).hexdigest()
full_hash = hashlib.sha1("\n".join(full_ids).encode("utf-8")).hexdigest()

pd.DataFrame({"Id": quick_ids}).to_csv(QUICK_EVAL_DIR / "quick20_sample_ids.csv", index=False)
pd.DataFrame({"Id": full_ids}).to_csv(FULL_EVAL_DIR / "matched100_sample_ids.csv", index=False)

# Deterministic training schedule; resume does not need RNG state.
schedule_rng = np.random.default_rng(SEED + 2026)
train_schedule = []
while len(train_schedule) < TRAIN_STEPS:
    permutation = schedule_rng.permutation(len(training_df)).tolist()
    train_schedule.extend(permutation)
train_schedule = train_schedule[:TRAIN_STEPS]

print("train / validation / test:", len(training_df), len(validation_df), len(test_df))
print("quick20 hash:", quick_hash)
print("matched100 hash:", full_hash)
print("quick20 first IDs:", quick_ids[:10])


In [ ]:

# 4) Load checkpoint-1200 (or latest resume checkpoint) as a trainable 4-bit LoRA model
for _name in ["model", "base_model", "processor", "optimizer"]:
    if _name in globals():
        del globals()[_name]
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

processor = AutoProcessor.from_pretrained(
    MODEL_ID,
    min_pixels=MIN_PIXELS,
    max_pixels=MAX_PIXELS,
    local_files_only=MODEL_LOCAL_FILES_ONLY,
    trust_remote_code=True,
    low_cpu_mem_usage=True,
)
processor.tokenizer.padding_side = "right"
if processor.tokenizer.pad_token is None:
    processor.tokenizer.pad_token = processor.tokenizer.eos_token

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.float16,
)

base_model = Qwen2VLForConditionalGeneration.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    torch_dtype=torch.float16,
    device_map="auto",
    local_files_only=MODEL_LOCAL_FILES_ONLY,
    trust_remote_code=True,
    low_cpu_mem_usage=True,
)
base_model.config.use_cache = False
base_model = prepare_model_for_kbit_training(base_model, use_gradient_checkpointing=True)

model = PeftModel.from_pretrained(
    base_model,
    START_ADAPTER_DIR,
    is_trainable=True,
)
model.config.use_cache = False
model.gradient_checkpointing_enable(gradient_checkpointing_kwargs={"use_reentrant": False})
model.enable_input_require_grads()

# Recompute and KV distribution scoring must represent the same deterministic policy.
for module in model.modules():
    if isinstance(module, torch.nn.Dropout):
        module.p = 0.0

model.print_trainable_parameters()

optimizer = bnb.optim.PagedAdamW8bit(
    [parameter for parameter in model.parameters() if parameter.requires_grad],
    lr=LEARNING_RATE,
)

optimizer_state_path = Path(START_ADAPTER_DIR) / "optimizer.pt"
if START_STEP > 0 and optimizer_state_path.exists():
    print("Loading optimizer state:", optimizer_state_path)
    optimizer.load_state_dict(torch.load(optimizer_state_path, map_location="cpu"))


def model_device(active_model):
    return next(active_model.parameters()).device


def batch_to_device(inputs, active_model):
    return {
        key: value.to(model_device(active_model)) if torch.is_tensor(value) else value
        for key, value in inputs.items()
    }

print_runtime_storage()


In [ ]:

# 5) Exact independent scoring and shared-prefix KV-cache scoring

def order_logprob_scores_recompute(
    active_model,
    example,
    candidate_orders,
    batch_size=1,
    grad_enabled=False,
):
    """Mean answer-token log probability for each candidate order."""
    old_padding_side = processor.tokenizer.padding_side
    processor.tokenizer.padding_side = "right"
    context = torch.enable_grad() if grad_enabled else torch.no_grad()
    try:
        prompt = processor.apply_chat_template(
            make_messages(example, include_answer=False),
            tokenize=False,
            add_generation_prompt=True,
        )
        images = [load_rgb(path) for path in example["image_paths"]]
        prompt_inputs = processor(text=[prompt], images=[images], return_tensors="pt")
        prompt_len = int(prompt_inputs["attention_mask"][0].sum().item())
        orders = list(candidate_orders)
        scores = []

        with context:
            for start in range(0, len(orders), int(batch_size)):
                batch_orders = orders[start:start + int(batch_size)]
                texts = [prompt + compact_order(order) for order in batch_orders]
                batch_images = [images for _ in batch_orders]
                inputs = processor(text=texts, images=batch_images, padding=True, return_tensors="pt")
                inputs = batch_to_device(inputs, active_model)
                outputs = active_model(**inputs, use_cache=False, return_dict=True)

                for row_index, _ in enumerate(batch_orders):
                    input_ids = inputs["input_ids"][row_index]
                    attention_len = int(inputs["attention_mask"][row_index].sum().item())
                    target_len = attention_len - prompt_len
                    target_ids = input_ids[prompt_len:prompt_len + target_len]
                    logits = outputs.logits[
                        row_index,
                        prompt_len - 1:prompt_len - 1 + target_len,
                    ]
                    token_log_probs = torch.log_softmax(logits.float(), dim=-1).gather(
                        1, target_ids[:, None]
                    ).squeeze(1)
                    scores.append(token_log_probs.mean())

                if not grad_enabled:
                    del outputs, inputs
        return torch.stack(scores)
    finally:
        processor.tokenizer.padding_side = old_padding_side


def target_ids_after_prompt(prompt, answer_text, device):
    tokenizer = processor.tokenizer
    prompt_ids = tokenizer(prompt, add_special_tokens=False)["input_ids"]
    full_ids = tokenizer(prompt + answer_text, add_special_tokens=False)["input_ids"]
    if full_ids[:len(prompt_ids)] != prompt_ids:
        raise RuntimeError(
            "Prompt/answer boundary tokenization changed. "
            f"prompt_tail={prompt_ids[-10:]}, full_prefix_tail={full_ids[max(0, len(prompt_ids)-10):len(prompt_ids)]}"
        )
    target_ids = full_ids[len(prompt_ids):]
    if not target_ids:
        raise RuntimeError(f"No target tokens for answer: {answer_text!r}")
    return torch.tensor(target_ids, dtype=torch.long, device=device)


def build_qwen2vl_continuation_position_ids(
    prompt_attention_mask,
    rope_deltas,
    continuation_length,
    device,
):
    if rope_deltas is None:
        raise RuntimeError("Prompt output has no rope_deltas.")
    batch_size = prompt_attention_mask.shape[0]
    prompt_text_lengths = prompt_attention_mask.long().sum(dim=1).to(device)
    offsets = torch.arange(continuation_length, dtype=torch.long, device=device)
    text_positions = prompt_text_lengths[:, None] + offsets[None, :]
    rope_deltas = rope_deltas.to(device).reshape(batch_size, 1)
    mrope_positions = text_positions + rope_deltas
    return mrope_positions.unsqueeze(0).expand(3, -1, -1).contiguous()


@torch.no_grad()
def order_logprob_scores_kv(active_model, example, candidate_orders):
    """Score all candidates after one common multimodal-prefix forward."""
    old_padding_side = processor.tokenizer.padding_side
    processor.tokenizer.padding_side = "right"
    prompt_cache = None
    base_cache_length = None
    try:
        device = model_device(active_model)
        prompt = processor.apply_chat_template(
            make_messages(example, include_answer=False),
            tokenize=False,
            add_generation_prompt=True,
        )
        images = [load_rgb(path) for path in example["image_paths"]]
        prompt_inputs = processor(text=[prompt], images=[images], return_tensors="pt")
        prompt_inputs = batch_to_device(prompt_inputs, active_model)

        prompt_outputs = active_model(**prompt_inputs, use_cache=True, return_dict=True)
        prompt_cache = prompt_outputs.past_key_values
        if prompt_cache is None:
            raise RuntimeError("Model did not return past_key_values.")
        if not hasattr(prompt_cache, "get_seq_length") or not hasattr(prompt_cache, "crop"):
            raise RuntimeError(f"Unsupported mutable cache type: {type(prompt_cache)}")

        base_cache_length = int(prompt_cache.get_seq_length())
        prompt_attention_mask = prompt_inputs["attention_mask"]
        prompt_len = int(prompt_attention_mask[0].sum().item())
        first_logits = prompt_outputs.logits[0, prompt_len - 1]
        prompt_rope_deltas = getattr(prompt_outputs, "rope_deltas", None)
        scores = []

        for order in candidate_orders:
            prompt_cache.crop(base_cache_length)
            target_ids = target_ids_after_prompt(prompt, compact_order(order), device)
            candidate_logits = [first_logits]

            if target_ids.numel() > 1:
                previous_ids = target_ids[:-1].unsqueeze(0)
                continuation_mask = torch.ones(
                    (1, previous_ids.shape[1]),
                    dtype=prompt_attention_mask.dtype,
                    device=device,
                )
                attention_mask = torch.cat([prompt_attention_mask, continuation_mask], dim=1)
                cache_position = torch.arange(
                    base_cache_length,
                    base_cache_length + previous_ids.shape[1],
                    dtype=torch.long,
                    device=device,
                )
                position_ids = build_qwen2vl_continuation_position_ids(
                    prompt_attention_mask=prompt_attention_mask,
                    rope_deltas=prompt_rope_deltas,
                    continuation_length=previous_ids.shape[1],
                    device=device,
                )
                continuation_outputs = active_model(
                    input_ids=previous_ids,
                    attention_mask=attention_mask,
                    position_ids=position_ids,
                    past_key_values=prompt_cache,
                    cache_position=cache_position,
                    use_cache=True,
                    return_dict=True,
                    pixel_values=None,
                    pixel_values_videos=None,
                    image_grid_thw=None,
                    video_grid_thw=None,
                )
                candidate_logits.extend(
                    continuation_outputs.logits[0, token_index]
                    for token_index in range(continuation_outputs.logits.shape[1])
                )

            logits = torch.stack(candidate_logits[:target_ids.numel()], dim=0)
            token_log_probs = torch.log_softmax(logits.float(), dim=-1).gather(
                1, target_ids[:, None]
            ).squeeze(1)
            scores.append(token_log_probs.mean())

        return torch.stack(scores)
    finally:
        if prompt_cache is not None and base_cache_length is not None and hasattr(prompt_cache, "crop"):
            prompt_cache.crop(base_cache_length)
        processor.tokenizer.padding_side = old_padding_side


def normalized_entropy(probabilities, eps=1e-12):
    probabilities = probabilities.float()
    entropy = -(probabilities * torch.log(probabilities.clamp_min(eps))).sum()
    maximum = math.log(max(2, probabilities.numel()))
    return entropy, entropy / maximum


def distributions_from_scores(scores):
    probabilities = torch.softmax(scores.float() / DISTRIBUTION_TEMPERATURE, dim=0)
    device = probabilities.device
    first_probabilities = torch.zeros(4, dtype=probabilities.dtype, device=device)
    last_probabilities = torch.zeros(4, dtype=probabilities.dtype, device=device)
    endpoint_probabilities = torch.zeros(12, dtype=probabilities.dtype, device=device)
    endpoint_pairs = [(first, last) for first in range(1, 5) for last in range(1, 5) if first != last]
    endpoint_to_index = {pair: idx for idx, pair in enumerate(endpoint_pairs)}

    for index, order in enumerate(PERMUTATIONS):
        first_probabilities[order[0] - 1] += probabilities[index]
        last_probabilities[order[-1] - 1] += probabilities[index]
        endpoint_probabilities[endpoint_to_index[(order[0], order[-1])]] += probabilities[index]

    return {
        "order": probabilities,
        "first": first_probabilities,
        "last": last_probabilities,
        "endpoint": endpoint_probabilities,
        "endpoint_pairs": endpoint_pairs,
    }


In [ ]:

# 6) Validate recompute vs KV-cache on the same 24 orders

def distribution_diagnostics(scores):
    distributions = distributions_from_scores(scores)
    order_probs = distributions["order"]
    first_probs = distributions["first"]
    last_probs = distributions["last"]
    full_entropy, full_norm_entropy = normalized_entropy(order_probs)
    return {
        "order_probs": order_probs,
        "first_probs": first_probs,
        "last_probs": last_probs,
        "top1_index": int(torch.argmax(order_probs).item()),
        "full_entropy": float(full_entropy.item()),
        "full_normalized_entropy": float(full_norm_entropy.item()),
    }


def run_kv_equivalence_diagnostic(active_model, sample_count=KV_DIAGNOSTIC_SAMPLES):
    active_model.eval()
    diagnostic_rows = []
    for sample_index in range(min(sample_count, len(quick_eval_df))):
        row = quick_eval_df.iloc[sample_index]
        example = row_to_order_example(sample_index, row)

        recompute_scores = order_logprob_scores_recompute(
            active_model,
            example,
            PERMUTATIONS,
            batch_size=NO_GRAD_RECOMPUTE_BATCH_SIZE,
            grad_enabled=False,
        ).detach().float()
        kv_scores = order_logprob_scores_kv(active_model, example, PERMUTATIONS).detach().float()

        recompute_diag = distribution_diagnostics(recompute_scores)
        kv_diag = distribution_diagnostics(kv_scores)
        rank_recompute = torch.argsort(recompute_scores, descending=True)
        rank_kv = torch.argsort(kv_scores, descending=True)

        diagnostic_rows.append({
            "sample_index": sample_index,
            "sample_id": str(row["Id"]),
            "max_abs_score_diff": float(torch.max(torch.abs(recompute_scores - kv_scores)).item()),
            "mean_abs_score_diff": float(torch.mean(torch.abs(recompute_scores - kv_scores)).item()),
            "max_abs_order_probability_diff": float(torch.max(torch.abs(
                recompute_diag["order_probs"] - kv_diag["order_probs"]
            )).item()),
            "max_abs_first_probability_diff": float(torch.max(torch.abs(
                recompute_diag["first_probs"] - kv_diag["first_probs"]
            )).item()),
            "max_abs_last_probability_diff": float(torch.max(torch.abs(
                recompute_diag["last_probs"] - kv_diag["last_probs"]
            )).item()),
            "top1_same": int(recompute_diag["top1_index"] == kv_diag["top1_index"]),
            "full_ranking_same": int(torch.equal(rank_recompute, rank_kv)),
            "full_entropy_abs_diff": abs(recompute_diag["full_entropy"] - kv_diag["full_entropy"]),
            "full_normalized_entropy_abs_diff": abs(
                recompute_diag["full_normalized_entropy"] - kv_diag["full_normalized_entropy"]
            ),
        })
        del recompute_scores, kv_scores
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    frame = pd.DataFrame(diagnostic_rows)
    out_path = EVAL_DIR / "kv_vs_recompute_full24_diagnostic.csv"
    frame.to_csv(out_path, index=False)
    display(frame)

    verified = bool(
        len(frame) > 0
        and frame["top1_same"].eq(1).all()
        and frame["max_abs_score_diff"].max() <= KV_SCORE_TOLERANCE
        and frame["max_abs_order_probability_diff"].max() <= KV_PROB_TOLERANCE
    )
    print("KV verified:", verified)
    print("Saved:", out_path)
    return frame, verified


kv_diagnostic_df, KV_VERIFIED = run_kv_equivalence_diagnostic(model)
USE_KV_DISTRIBUTION_SCORER = bool(PREFER_KV_FOR_DISTRIBUTION and KV_VERIFIED)
print("Distribution scorer:", "KV-cache" if USE_KV_DISTRIBUTION_SCORER else "independent recompute")


def score_full24_distribution(active_model, example):
    active_model.eval()
    if USE_KV_DISTRIBUTION_SCORER:
        return order_logprob_scores_kv(active_model, example, PERMUTATIONS).detach().float()
    return order_logprob_scores_recompute(
        active_model,
        example,
        PERMUTATIONS,
        batch_size=NO_GRAD_RECOMPUTE_BATCH_SIZE,
        grad_enabled=False,
    ).detach().float()


In [ ]:

# 7) Full-24 multilevel objective and exact score-space chain-rule backward

def one_hot(index, classes, device, dtype):
    target = torch.zeros(classes, device=device, dtype=dtype)
    target[int(index)] = 1.0
    return target


def multilevel_objective_from_scores(scores, gold_order):
    distributions = distributions_from_scores(scores)
    order_probs = distributions["order"]
    first_probs = distributions["first"]
    last_probs = distributions["last"]

    gold_tuple = tuple(int(value) for value in gold_order)
    gold_order_index = PERMUTATION_TO_INDEX[gold_tuple]
    gold_first_index = gold_tuple[0] - 1
    gold_last_index = gold_tuple[-1] - 1
    eps = 1e-12

    order_ce = -torch.log(order_probs[gold_order_index].clamp_min(eps))
    first_ce = -torch.log(first_probs[gold_first_index].clamp_min(eps))
    last_ce = -torch.log(last_probs[gold_last_index].clamp_min(eps))

    order_target = one_hot(gold_order_index, 24, scores.device, order_probs.dtype)
    first_target = one_hot(gold_first_index, 4, scores.device, first_probs.dtype)
    last_target = one_hot(gold_last_index, 4, scores.device, last_probs.dtype)

    order_brier = torch.sum((order_probs - order_target) ** 2)
    first_brier = torch.sum((first_probs - first_target) ** 2)
    last_brier = torch.sum((last_probs - last_target) ** 2)

    components = {
        "order_ce": order_ce,
        "first_ce": first_ce,
        "last_ce": last_ce,
        "order_brier": order_brier,
        "first_brier": first_brier,
        "last_brier": last_brier,
    }
    total = sum(float(LOSS_WEIGHTS[name]) * value for name, value in components.items())
    return total, components, distributions


def score_space_gradient_coefficients(detached_scores, gold_order):
    score_leaf = detached_scores.detach().float().clone().requires_grad_(True)
    total_loss, components, distributions = multilevel_objective_from_scores(score_leaf, gold_order)
    coefficients = torch.autograd.grad(total_loss, score_leaf, retain_graph=False)[0].detach()
    component_values = {name: float(value.detach().cpu().item()) for name, value in components.items()}
    component_values["total_loss"] = float(total_loss.detach().cpu().item())
    return coefficients, component_values, {
        name: value.detach() for name, value in distributions.items() if name != "endpoint_pairs"
    }


def backward_full24_exact(active_model, example, coefficients, reference_scores):
    """
    Exact chain rule:
        dL/dtheta = sum_i (dL/dscore_i) * dscore_i/dtheta

    The common text/PIL inputs are prepared once, while only a small
    candidate-chunk graph is alive at any moment.
    """
    active_model.train()
    old_padding_side = processor.tokenizer.padding_side
    processor.tokenizer.padding_side = "right"
    rescored_values = []
    try:
        prompt = processor.apply_chat_template(
            make_messages(example, include_answer=False),
            tokenize=False,
            add_generation_prompt=True,
        )
        images = [load_rgb(path) for path in example["image_paths"]]
        prompt_inputs = processor(text=[prompt], images=[images], return_tensors="pt")
        prompt_len = int(prompt_inputs["attention_mask"][0].sum().item())
        orders = list(PERMUTATIONS)

        for start in range(0, len(orders), GRAD_RESCORE_BATCH_SIZE):
            batch_orders = orders[start:start + GRAD_RESCORE_BATCH_SIZE]
            batch_coefficients = coefficients[start:start + len(batch_orders)].to(model_device(active_model))
            texts = [prompt + compact_order(order) for order in batch_orders]
            batch_images = [images for _ in batch_orders]
            inputs = processor(text=texts, images=batch_images, padding=True, return_tensors="pt")
            inputs = batch_to_device(inputs, active_model)
            outputs = active_model(**inputs, use_cache=False, return_dict=True)

            batch_scores = []
            for row_index, _ in enumerate(batch_orders):
                input_ids = inputs["input_ids"][row_index]
                attention_len = int(inputs["attention_mask"][row_index].sum().item())
                target_len = attention_len - prompt_len
                target_ids = input_ids[prompt_len:prompt_len + target_len]
                logits = outputs.logits[
                    row_index,
                    prompt_len - 1:prompt_len - 1 + target_len,
                ]
                token_log_probs = torch.log_softmax(logits.float(), dim=-1).gather(
                    1, target_ids[:, None]
                ).squeeze(1)
                batch_scores.append(token_log_probs.mean())

            batch_scores = torch.stack(batch_scores)
            surrogate = torch.sum(batch_coefficients.to(batch_scores.dtype) * batch_scores)
            surrogate.backward()
            rescored_values.extend(batch_scores.detach().float().cpu().tolist())
            del outputs, inputs, batch_scores, surrogate

        rescored_tensor = torch.tensor(rescored_values, dtype=torch.float32)
        reference_cpu = reference_scores.detach().float().cpu()
        max_abs_difference = float(torch.max(torch.abs(rescored_tensor - reference_cpu)).item())
        return max_abs_difference, rescored_tensor
    finally:
        processor.tokenizer.padding_side = old_padding_side


def probability_margin(probabilities):
    values = torch.sort(probabilities.float(), descending=True).values
    return float((values[0] - values[1]).item()), float(values[0].item()), float(values[1].item())


def training_distribution_log(scores, gold_order):
    total, components, distributions = multilevel_objective_from_scores(scores.detach().float(), gold_order)
    order_probs = distributions["order"]
    first_probs = distributions["first"]
    last_probs = distributions["last"]
    full_entropy, full_norm_entropy = normalized_entropy(order_probs)
    first_entropy, first_norm_entropy = normalized_entropy(first_probs)
    last_entropy, last_norm_entropy = normalized_entropy(last_probs)
    full_margin, full_top1, full_top2 = probability_margin(order_probs)
    first_margin, first_top1, first_top2 = probability_margin(first_probs)
    last_margin, last_top1, last_top2 = probability_margin(last_probs)
    gold_tuple = tuple(int(x) for x in gold_order)
    gold_index = PERMUTATION_TO_INDEX[gold_tuple]
    return {
        "pre_total_loss": float(total.item()),
        "pre_gold_order_probability": float(order_probs[gold_index].item()),
        "pre_full_top1_probability": full_top1,
        "pre_full_top2_probability": full_top2,
        "pre_full_top1_margin": full_margin,
        "pre_full_normalized_entropy": float(full_norm_entropy.item()),
        "pre_first_gold_probability": float(first_probs[gold_tuple[0] - 1].item()),
        "pre_first_top1_probability": first_top1,
        "pre_first_top2_probability": first_top2,
        "pre_first_marginal_margin": first_margin,
        "pre_first_normalized_entropy": float(first_norm_entropy.item()),
        "pre_last_gold_probability": float(last_probs[gold_tuple[-1] - 1].item()),
        "pre_last_top1_probability": last_top1,
        "pre_last_top2_probability": last_top2,
        "pre_last_marginal_margin": last_margin,
        "pre_last_normalized_entropy": float(last_norm_entropy.item()),
    }


In [ ]:
# 8) Evaluation: full-24 calibration + pairwise / conditional agreement metrics

def order_metrics(pred_order, gold_order):
    pred_order = [int(x) for x in pred_order]
    gold_order = [int(x) for x in gold_order]
    pred_rank = {frame: idx for idx, frame in enumerate(pred_order)}
    gold_rank = {frame: idx for idx, frame in enumerate(gold_order)}
    return {
        "exact": float(pred_order == gold_order),
        "first_correct": float(pred_order[0] == gold_order[0]),
        "last_correct": float(pred_order[-1] == gold_order[-1]),
        "both_endpoints_correct": float(
            pred_order[0] == gold_order[0] and pred_order[-1] == gold_order[-1]
        ),
        "position_accuracy": float(np.mean([p == g for p, g in zip(pred_order, gold_order)])),
        "pair_accuracy": float(np.mean([
            (pred_rank[a] < pred_rank[b]) == (gold_rank[a] < gold_rank[b])
            for a, b in itertools.combinations([1, 2, 3, 4], 2)
        ])),
    }


def binary_auc(labels, scores):
    labels = np.asarray(labels, dtype=np.int64)
    scores = np.asarray(scores, dtype=np.float64)
    valid = np.isfinite(scores)
    labels = labels[valid]
    scores = scores[valid]
    positives = int(labels.sum())
    negatives = int(len(labels) - positives)
    if positives == 0 or negatives == 0:
        return float("nan")
    ranks = pd.Series(scores).rank(method="average").to_numpy()
    positive_rank_sum = float(ranks[labels == 1].sum())
    return (positive_rank_sum - positives * (positives + 1) / 2.0) / (positives * negatives)


def select_entropy_largest_gap(values, min_fraction=0.10, max_fraction=0.50, default_fraction=0.40, min_gap_z=3.0):
    entropy = np.asarray(values, dtype=np.float64)
    eps = 1e-8
    risk = -np.log(1.0 - np.clip(entropy, 0.0, 1.0 - eps) + eps)
    order = np.argsort(-risk)
    sorted_risk = risk[order]
    if len(sorted_risk) < 3:
        return np.ones(len(sorted_risk), dtype=bool), float("-inf"), "all", float("nan")
    gaps = sorted_risk[:-1] - sorted_risk[1:]
    n = len(risk)
    min_count = max(1, int(np.ceil(n * min_fraction)))
    max_count = min(n - 1, int(np.floor(n * max_fraction)))
    start = min_count - 1
    end = max_count - 1
    candidate = gaps[start:end + 1]
    gap_index = start + int(np.argmax(candidate))
    gap_median = np.median(gaps)
    gap_mad = np.median(np.abs(gaps - gap_median))
    gap_z = float((gaps[gap_index] - gap_median) / (1.4826 * gap_mad + eps))
    if gap_z >= min_gap_z:
        threshold = float((sorted_risk[gap_index] + sorted_risk[gap_index + 1]) / 2.0)
        method = "largest_gap"
    else:
        threshold = float(np.quantile(risk, 1.0 - default_fraction))
        method = "fixed_quantile"
    return risk >= threshold, threshold, method, gap_z


def pairwise_views_from_order_probs(order_probs):
    """
    Derive P(i before j) from the same full-24 distribution.
    No extra model call is made.
    """
    probs = order_probs.detach().float().cpu()
    before = torch.zeros((4, 4), dtype=probs.dtype)
    for order_index, order in enumerate(PERMUTATIONS):
        probability = probs[order_index]
        positions = {frame: position for position, frame in enumerate(order)}
        for first_frame in range(1, 5):
            for second_frame in range(1, 5):
                if first_frame != second_frame and positions[first_frame] < positions[second_frame]:
                    before[first_frame - 1, second_frame - 1] += probability

    # Borda-style endpoint evidence. Divide by 3 so every score lies in [0, 1].
    early_scores = before.sum(dim=1) / 3.0
    late_scores = before.sum(dim=0) / 3.0
    early_sorted = torch.argsort(early_scores, descending=True)
    late_sorted = torch.argsort(late_scores, descending=True)

    hard_before = before > 0.5
    cycle_count = 0
    for a, b, c in itertools.combinations(range(4), 3):
        clockwise = bool(hard_before[a, b] and hard_before[b, c] and hard_before[c, a])
        counterclockwise = bool(hard_before[b, a] and hard_before[c, b] and hard_before[a, c])
        cycle_count += int(clockwise or counterclockwise)

    return {
        "before": before,
        "early_scores": early_scores,
        "late_scores": late_scores,
        "first_prediction": int(early_sorted[0].item()) + 1,
        "last_prediction": int(late_sorted[0].item()) + 1,
        "first_margin": float(early_scores[early_sorted[0]].item() - early_scores[early_sorted[1]].item()),
        "last_margin": float(late_scores[late_sorted[0]].item() - late_scores[late_sorted[1]].item()),
        "cycle_count": int(cycle_count),
        "transitivity_score": float(1.0 - cycle_count / 4.0),
    }


def conditional_endpoint_views(order_probs, marginal_first_prediction, marginal_last_prediction):
    """Conditional endpoint distributions derived from full-24 probabilities."""
    probs = order_probs.detach().float().cpu()
    first_given_last = torch.zeros(4, dtype=probs.dtype)
    last_given_first = torch.zeros(4, dtype=probs.dtype)

    for order_index, order in enumerate(PERMUTATIONS):
        probability = probs[order_index]
        if order[-1] == int(marginal_last_prediction):
            first_given_last[order[0] - 1] += probability
        if order[0] == int(marginal_first_prediction):
            last_given_first[order[-1] - 1] += probability

    eps = 1e-12
    if float(first_given_last.sum().item()) > eps:
        first_given_last = first_given_last / first_given_last.sum()
    else:
        first_given_last.fill_(0.25)
    if float(last_given_first.sum().item()) > eps:
        last_given_first = last_given_first / last_given_first.sum()
    else:
        last_given_first.fill_(0.25)

    first_sorted = torch.argsort(first_given_last, descending=True)
    last_sorted = torch.argsort(last_given_first, descending=True)
    first_entropy, first_norm_entropy = normalized_entropy(first_given_last)
    last_entropy, last_norm_entropy = normalized_entropy(last_given_first)

    return {
        "first_probs": first_given_last,
        "last_probs": last_given_first,
        "first_prediction": int(first_sorted[0].item()) + 1,
        "last_prediction": int(last_sorted[0].item()) + 1,
        "first_margin": float(first_given_last[first_sorted[0]].item() - first_given_last[first_sorted[1]].item()),
        "last_margin": float(last_given_first[last_sorted[0]].item() - last_given_first[last_sorted[1]].item()),
        "first_entropy": float(first_entropy.item()),
        "first_normalized_entropy": float(first_norm_entropy.item()),
        "last_entropy": float(last_entropy.item()),
        "last_normalized_entropy": float(last_norm_entropy.item()),
    }


def vote_summary(values, preferred_value):
    values = [int(value) for value in values]
    counts = {value: values.count(value) for value in sorted(set(values))}
    max_votes = max(counts.values())
    winners = [value for value, count in counts.items() if count == max_votes]
    prediction = int(preferred_value) if int(preferred_value) in winners else int(winners[0])
    return {
        "prediction": prediction,
        "vote_count": int(max_votes),
        "all_agree": int(max_votes == len(values)),
        "unique_majority": int(max_votes >= 2),
        "disagree_count": int(len(values) - max_votes),
    }


def masked_mean(frame, mask, column):
    return frame.loc[mask, column].mean() if bool(np.asarray(mask).any()) else float("nan")


def detail_from_scores(row, scores):
    gold_order = order_to_sequence(row["Answer_list"])
    gold_tuple = tuple(gold_order)
    gold_index = PERMUTATION_TO_INDEX[gold_tuple]
    distributions = distributions_from_scores(scores)
    order_probs = distributions["order"].detach().float().cpu()
    first_probs = distributions["first"].detach().float().cpu()
    last_probs = distributions["last"].detach().float().cpu()
    endpoint_probs = distributions["endpoint"].detach().float().cpu()
    endpoint_pairs = distributions["endpoint_pairs"]

    top_indices = torch.argsort(order_probs, descending=True)
    top1_index = int(top_indices[0].item())
    top2_index = int(top_indices[1].item())
    pred_order = list(PERMUTATIONS[top1_index])
    metrics = order_metrics(pred_order, gold_order)

    full_entropy, full_norm_entropy = normalized_entropy(order_probs)
    first_entropy, first_norm_entropy = normalized_entropy(first_probs)
    last_entropy, last_norm_entropy = normalized_entropy(last_probs)
    endpoint_entropy, endpoint_norm_entropy = normalized_entropy(endpoint_probs)

    first_sorted = torch.argsort(first_probs, descending=True)
    last_sorted = torch.argsort(last_probs, descending=True)
    endpoint_sorted = torch.argsort(endpoint_probs, descending=True)
    marginal_first_prediction = int(first_sorted[0].item()) + 1
    marginal_last_prediction = int(last_sorted[0].item()) + 1

    pairwise = pairwise_views_from_order_probs(order_probs)
    conditional = conditional_endpoint_views(
        order_probs,
        marginal_first_prediction=marginal_first_prediction,
        marginal_last_prediction=marginal_last_prediction,
    )
    first_vote = vote_summary(
        [marginal_first_prediction, pairwise["first_prediction"], conditional["first_prediction"]],
        preferred_value=marginal_first_prediction,
    )
    last_vote = vote_summary(
        [marginal_last_prediction, pairwise["last_prediction"], conditional["last_prediction"]],
        preferred_value=marginal_last_prediction,
    )

    first_order_pairwise_disagree = int(marginal_first_prediction != pairwise["first_prediction"])
    last_order_pairwise_disagree = int(marginal_last_prediction != pairwise["last_prediction"])
    endpoint_disagree_count = first_order_pairwise_disagree + last_order_pairwise_disagree

    wrong_mask = torch.ones(24, dtype=torch.bool)
    wrong_mask[gold_index] = False
    strongest_wrong_probability = float(order_probs[wrong_mask].max().item())

    order_target = torch.zeros(24); order_target[gold_index] = 1.0
    first_target = torch.zeros(4); first_target[gold_tuple[0] - 1] = 1.0
    last_target = torch.zeros(4); last_target[gold_tuple[-1] - 1] = 1.0

    record = {
        "sample_id": str(row["Id"]),
        "gold_order": compact_order(gold_order),
        "pred_order": compact_order(pred_order),
        **metrics,
        "gold_order_probability": float(order_probs[gold_index].item()),
        "gold_order_rank": int((top_indices == gold_index).nonzero(as_tuple=False)[0].item()) + 1,
        "gold_order_nll": float(-math.log(max(float(order_probs[gold_index].item()), 1e-12))),
        "gold_minus_strongest_wrong_probability": float(order_probs[gold_index].item()) - strongest_wrong_probability,
        "full_top1_probability": float(order_probs[top1_index].item()),
        "full_top2_probability": float(order_probs[top2_index].item()),
        "full_top1_margin": float(order_probs[top1_index].item() - order_probs[top2_index].item()),
        "full_entropy": float(full_entropy.item()),
        "full_normalized_entropy": float(full_norm_entropy.item()),
        "order_brier": float(torch.sum((order_probs - order_target) ** 2).item()),

        # Direct top-1 endpoints versus marginal endpoints.
        "direct_top1_first_prediction": int(pred_order[0]),
        "direct_top1_last_prediction": int(pred_order[-1]),
        "direct_vs_marginal_first_disagree": int(pred_order[0] != marginal_first_prediction),
        "direct_vs_marginal_last_disagree": int(pred_order[-1] != marginal_last_prediction),

        # Full-24 marginal endpoints.
        "first_prediction": marginal_first_prediction,
        "first_gold_probability": float(first_probs[gold_tuple[0] - 1].item()),
        "first_top1_probability": float(first_probs[first_sorted[0]].item()),
        "first_top2_probability": float(first_probs[first_sorted[1]].item()),
        "first_marginal_margin": float(first_probs[first_sorted[0]].item() - first_probs[first_sorted[1]].item()),
        "first_entropy": float(first_entropy.item()),
        "first_normalized_entropy": float(first_norm_entropy.item()),
        "first_brier": float(torch.sum((first_probs - first_target) ** 2).item()),
        "last_prediction": marginal_last_prediction,
        "last_gold_probability": float(last_probs[gold_tuple[-1] - 1].item()),
        "last_top1_probability": float(last_probs[last_sorted[0]].item()),
        "last_top2_probability": float(last_probs[last_sorted[1]].item()),
        "last_marginal_margin": float(last_probs[last_sorted[0]].item() - last_probs[last_sorted[1]].item()),
        "last_entropy": float(last_entropy.item()),
        "last_normalized_entropy": float(last_norm_entropy.item()),
        "last_brier": float(torch.sum((last_probs - last_target) ** 2).item()),

        # Pairwise endpoint evidence derived from full-24 probabilities.
        "pairwise_first_prediction": pairwise["first_prediction"],
        "pairwise_first_correct": int(pairwise["first_prediction"] == gold_tuple[0]),
        "pairwise_first_margin": pairwise["first_margin"],
        "pairwise_last_prediction": pairwise["last_prediction"],
        "pairwise_last_correct": int(pairwise["last_prediction"] == gold_tuple[-1]),
        "pairwise_last_margin": pairwise["last_margin"],
        "pairwise_both_endpoints_correct": int(
            pairwise["first_prediction"] == gold_tuple[0]
            and pairwise["last_prediction"] == gold_tuple[-1]
        ),
        "pairwise_cycle_count": pairwise["cycle_count"],
        "pairwise_transitivity_score": pairwise["transitivity_score"],

        # Marginal vs pairwise disagreement routing signals.
        "order_pairwise_first_disagree": first_order_pairwise_disagree,
        "order_pairwise_last_disagree": last_order_pairwise_disagree,
        "order_pairwise_endpoint_disagree_count": endpoint_disagree_count,
        "order_pairwise_any_endpoint_disagree": int(endpoint_disagree_count >= 1),
        "order_pairwise_both_endpoints_disagree": int(endpoint_disagree_count == 2),

        # Conditional endpoints from the same full-24 distribution.
        "conditional_first_given_marginal_last_prediction": conditional["first_prediction"],
        "conditional_first_correct": int(conditional["first_prediction"] == gold_tuple[0]),
        "conditional_first_margin": conditional["first_margin"],
        "conditional_first_normalized_entropy": conditional["first_normalized_entropy"],
        "conditional_last_given_marginal_first_prediction": conditional["last_prediction"],
        "conditional_last_correct": int(conditional["last_prediction"] == gold_tuple[-1]),
        "conditional_last_margin": conditional["last_margin"],
        "conditional_last_normalized_entropy": conditional["last_normalized_entropy"],

        # Three-way marginal / pairwise / conditional agreement.
        "first_threeway_majority_prediction": first_vote["prediction"],
        "first_threeway_majority_correct": int(first_vote["prediction"] == gold_tuple[0]),
        "first_threeway_vote_count": first_vote["vote_count"],
        "first_threeway_all_agree": first_vote["all_agree"],
        "first_threeway_unique_majority": first_vote["unique_majority"],
        "first_threeway_disagree_count": first_vote["disagree_count"],
        "last_threeway_majority_prediction": last_vote["prediction"],
        "last_threeway_majority_correct": int(last_vote["prediction"] == gold_tuple[-1]),
        "last_threeway_vote_count": last_vote["vote_count"],
        "last_threeway_all_agree": last_vote["all_agree"],
        "last_threeway_unique_majority": last_vote["unique_majority"],
        "last_threeway_disagree_count": last_vote["disagree_count"],
        "threeway_both_endpoints_all_agree": int(first_vote["all_agree"] and last_vote["all_agree"]),

        "endpoint_pair_prediction": f"{endpoint_pairs[int(endpoint_sorted[0].item())][0]}-{endpoint_pairs[int(endpoint_sorted[0].item())][1]}",
        "endpoint_pair_top1_probability": float(endpoint_probs[endpoint_sorted[0]].item()),
        "endpoint_pair_top2_probability": float(endpoint_probs[endpoint_sorted[1]].item()),
        "endpoint_pair_margin": float(endpoint_probs[endpoint_sorted[0]].item() - endpoint_probs[endpoint_sorted[1]].item()),
        "endpoint_pair_entropy": float(endpoint_entropy.item()),
        "endpoint_pair_normalized_entropy": float(endpoint_norm_entropy.item()),
    }

    if SAVE_ALL_24_PROBABILITIES:
        for index, order in enumerate(PERMUTATIONS):
            record[f"p_order_{''.join(map(str, order))}"] = float(order_probs[index].item())
    for frame in range(1, 5):
        record[f"p_first_{frame}"] = float(first_probs[frame - 1].item())
        record[f"p_last_{frame}"] = float(last_probs[frame - 1].item())
        record[f"pairwise_early_score_{frame}"] = float(pairwise["early_scores"][frame - 1].item())
        record[f"pairwise_late_score_{frame}"] = float(pairwise["late_scores"][frame - 1].item())
        record[f"p_conditional_first_given_marginal_last_{frame}"] = float(conditional["first_probs"][frame - 1].item())
        record[f"p_conditional_last_given_marginal_first_{frame}"] = float(conditional["last_probs"][frame - 1].item())
    for first_frame, second_frame in itertools.combinations(range(1, 5), 2):
        record[f"p_pair_{first_frame}_before_{second_frame}"] = float(
            pairwise["before"][first_frame - 1, second_frame - 1].item()
        )
    return record


def build_eval_summary(detail_df, tag, gate_metadata):
    global_mask = detail_df["global_low_confidence"].astype(bool)
    first_agree = detail_df["order_pairwise_first_disagree"].eq(0)
    first_disagree = ~first_agree
    last_agree = detail_df["order_pairwise_last_disagree"].eq(0)
    last_disagree = ~last_agree
    endpoint_disagree_count = detail_df["order_pairwise_endpoint_disagree_count"]

    rows = {
        "tag": tag,
        "rows": len(detail_df),
        "exact": detail_df["exact"].mean(),
        "first": detail_df["first_correct"].mean(),
        "last": detail_df["last_correct"].mean(),
        "both_endpoints": detail_df["both_endpoints_correct"].mean(),
        "position": detail_df["position_accuracy"].mean(),
        "pair": detail_df["pair_accuracy"].mean(),
        "mean_gold_order_probability": detail_df["gold_order_probability"].mean(),
        "mean_gold_order_rank": detail_df["gold_order_rank"].mean(),
        "mean_gold_order_nll": detail_df["gold_order_nll"].mean(),
        "mean_order_brier": detail_df["order_brier"].mean(),
        "mean_first_brier": detail_df["first_brier"].mean(),
        "mean_last_brier": detail_df["last_brier"].mean(),
        "mean_full_margin": detail_df["full_top1_margin"].mean(),
        "mean_full_normalized_entropy": detail_df["full_normalized_entropy"].mean(),
        "mean_first_margin": detail_df["first_marginal_margin"].mean(),
        "mean_first_normalized_entropy": detail_df["first_normalized_entropy"].mean(),
        "mean_last_margin": detail_df["last_marginal_margin"].mean(),
        "mean_last_normalized_entropy": detail_df["last_normalized_entropy"].mean(),
        "full_entropy_error_auc": binary_auc(1 - detail_df["exact"], detail_df["full_normalized_entropy"]),
        "full_margin_correct_auc": binary_auc(detail_df["exact"], detail_df["full_top1_margin"]),
        "first_entropy_error_auc": binary_auc(1 - detail_df["first_correct"], detail_df["first_normalized_entropy"]),
        "first_margin_correct_auc": binary_auc(detail_df["first_correct"], detail_df["first_marginal_margin"]),
        "last_entropy_error_auc": binary_auc(1 - detail_df["last_correct"], detail_df["last_normalized_entropy"]),
        "last_margin_correct_auc": binary_auc(detail_df["last_correct"], detail_df["last_marginal_margin"]),

        # Pairwise and conditional endpoint accuracy.
        "pairwise_first": detail_df["pairwise_first_correct"].mean(),
        "pairwise_last": detail_df["pairwise_last_correct"].mean(),
        "pairwise_both_endpoints": detail_df["pairwise_both_endpoints_correct"].mean(),
        "conditional_first": detail_df["conditional_first_correct"].mean(),
        "conditional_last": detail_df["conditional_last_correct"].mean(),
        "threeway_majority_first": detail_df["first_threeway_majority_correct"].mean(),
        "threeway_majority_last": detail_df["last_threeway_majority_correct"].mean(),
        "mean_pairwise_first_margin": detail_df["pairwise_first_margin"].mean(),
        "mean_pairwise_last_margin": detail_df["pairwise_last_margin"].mean(),
        "mean_pairwise_cycle_count": detail_df["pairwise_cycle_count"].mean(),
        "mean_pairwise_transitivity_score": detail_df["pairwise_transitivity_score"].mean(),

        # Disagreement rates and their error-selection strength.
        "order_pairwise_first_disagreement_rate": detail_df["order_pairwise_first_disagree"].mean(),
        "order_pairwise_last_disagreement_rate": detail_df["order_pairwise_last_disagree"].mean(),
        "order_pairwise_any_endpoint_disagreement_rate": detail_df["order_pairwise_any_endpoint_disagree"].mean(),
        "order_pairwise_both_endpoints_disagreement_rate": detail_df["order_pairwise_both_endpoints_disagree"].mean(),
        "first_disagreement_first_error_auc": binary_auc(
            1 - detail_df["first_correct"], detail_df["order_pairwise_first_disagree"]
        ),
        "last_disagreement_last_error_auc": binary_auc(
            1 - detail_df["last_correct"], detail_df["order_pairwise_last_disagree"]
        ),
        "any_endpoint_disagreement_exact_error_auc": binary_auc(
            1 - detail_df["exact"], detail_df["order_pairwise_any_endpoint_disagree"]
        ),
        "first_accuracy_when_order_pairwise_agree": masked_mean(detail_df, first_agree, "first_correct"),
        "first_accuracy_when_order_pairwise_disagree": masked_mean(detail_df, first_disagree, "first_correct"),
        "last_accuracy_when_order_pairwise_agree": masked_mean(detail_df, last_agree, "last_correct"),
        "last_accuracy_when_order_pairwise_disagree": masked_mean(detail_df, last_disagree, "last_correct"),
        "exact_when_endpoint_disagree_count_0": masked_mean(detail_df, endpoint_disagree_count.eq(0), "exact"),
        "exact_when_endpoint_disagree_count_1": masked_mean(detail_df, endpoint_disagree_count.eq(1), "exact"),
        "exact_when_endpoint_disagree_count_2": masked_mean(detail_df, endpoint_disagree_count.eq(2), "exact"),
        "count_endpoint_disagree_0": int(endpoint_disagree_count.eq(0).sum()),
        "count_endpoint_disagree_1": int(endpoint_disagree_count.eq(1).sum()),
        "count_endpoint_disagree_2": int(endpoint_disagree_count.eq(2).sum()),
        "first_threeway_all_agree_rate": detail_df["first_threeway_all_agree"].mean(),
        "last_threeway_all_agree_rate": detail_df["last_threeway_all_agree"].mean(),
        "both_endpoints_threeway_all_agree_rate": detail_df["threeway_both_endpoints_all_agree"].mean(),

        "global_low_count": int(global_mask.sum()),
        "global_low_exact": detail_df.loc[global_mask, "exact"].mean() if global_mask.any() else float("nan"),
        "global_low_first_entropy_error_auc": binary_auc(
            1 - detail_df.loc[global_mask, "first_correct"],
            detail_df.loc[global_mask, "first_normalized_entropy"],
        ) if global_mask.any() else float("nan"),
        "global_low_last_entropy_error_auc": binary_auc(
            1 - detail_df.loc[global_mask, "last_correct"],
            detail_df.loc[global_mask, "last_normalized_entropy"],
        ) if global_mask.any() else float("nan"),
        "global_low_order_pairwise_first_disagreement_rate": detail_df.loc[
            global_mask, "order_pairwise_first_disagree"
        ].mean() if global_mask.any() else float("nan"),
        "global_low_order_pairwise_last_disagreement_rate": detail_df.loc[
            global_mask, "order_pairwise_last_disagree"
        ].mean() if global_mask.any() else float("nan"),
        "global_low_any_endpoint_disagreement_exact_error_auc": binary_auc(
            1 - detail_df.loc[global_mask, "exact"],
            detail_df.loc[global_mask, "order_pairwise_any_endpoint_disagree"],
        ) if global_mask.any() else float("nan"),
        **gate_metadata,
    }

    for prefix, correct_column, margin_column, entropy_column in [
        ("full", "exact", "full_top1_margin", "full_normalized_entropy"),
        ("first", "first_correct", "first_marginal_margin", "first_normalized_entropy"),
        ("last", "last_correct", "last_marginal_margin", "last_normalized_entropy"),
    ]:
        correct_mask = detail_df[correct_column].eq(1)
        wrong_mask = ~correct_mask
        rows[f"{prefix}_correct_mean_margin"] = detail_df.loc[correct_mask, margin_column].mean()
        rows[f"{prefix}_wrong_mean_margin"] = detail_df.loc[wrong_mask, margin_column].mean()
        rows[f"{prefix}_correct_mean_normalized_entropy"] = detail_df.loc[correct_mask, entropy_column].mean()
        rows[f"{prefix}_wrong_mean_normalized_entropy"] = detail_df.loc[wrong_mask, entropy_column].mean()
    return pd.DataFrame([rows])


@torch.no_grad()
def evaluate_model(active_model, rows, tag, output_dir):
    active_model.eval()
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)
    details = []
    for row_index, row in tqdm(rows.iterrows(), total=len(rows), desc=f"eval {tag}"):
        example = row_to_order_example(row_index, row)
        scores = score_full24_distribution(active_model, example)
        details.append(detail_from_scores(row, scores))
        del scores
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    detail_df = pd.DataFrame(details)
    mask, threshold, method, gap_z = select_entropy_largest_gap(detail_df["full_normalized_entropy"].to_numpy())
    detail_df["global_low_confidence"] = mask.astype(int)
    gate_metadata = {
        "global_gate_method": method,
        "global_gate_risk_threshold": threshold,
        "global_gate_gap_z": gap_z,
    }
    summary_df = build_eval_summary(detail_df, tag, gate_metadata)
    detail_path = output_dir / f"{tag}_detail.csv"
    summary_path = output_dir / f"{tag}_summary.csv"
    detail_df.to_csv(detail_path, index=False)
    summary_df.to_csv(summary_path, index=False)
    display(summary_df.T)
    print("saved:", detail_path)
    print("saved:", summary_path)
    return detail_df, summary_df


In [ ]:
# 9) Baseline evaluation at step 0
initial_summary_df = None
if START_STEP == 0:
    if RUN_INITIAL_QUICK_EVAL:
        initial_detail_df, initial_summary_df = evaluate_model(
            model,
            quick_eval_df,
            tag="step_000_checkpoint1200_quick20",
            output_dir=QUICK_EVAL_DIR,
        )
    if RUN_INITIAL_MATCHED100_EVAL:
        initial_full_detail_df, initial_full_summary_df = evaluate_model(
            model,
            full_eval_df,
            tag="step_000_checkpoint1200_matched100",
            output_dir=FULL_EVAL_DIR,
        )
elif START_STEP > 0:
    print("Resume detected; skipping step-0 evaluation.")


In [ ]:

# 10) Train / save / resume
TRAIN_LOG_PATH = OUTPUT_DIR / "full24_train_step_log.csv"


def save_training_checkpoint(step):
    checkpoint_dir = CHECKPOINT_ROOT / f"checkpoint-1200-full24-step-{int(step):03d}"
    checkpoint_dir.mkdir(parents=True, exist_ok=True)
    model.save_pretrained(checkpoint_dir, safe_serialization=True)
    torch.save(optimizer.state_dict(), checkpoint_dir / "optimizer.pt")
    state = {
        "step": int(step),
        "base_checkpoint": str(BASE_CHECKPOINT_DIR),
        "loss_weights": LOSS_WEIGHTS,
        "distribution_scorer": "kv" if USE_KV_DISTRIBUTION_SCORER else "recompute",
    }
    with open(checkpoint_dir / "training_state.json", "w", encoding="utf-8") as handle:
        json.dump(state, handle, ensure_ascii=False, indent=2)
    print("saved checkpoint:", checkpoint_dir)
    return checkpoint_dir


def load_existing_train_log():
    if TRAIN_LOG_PATH.exists():
        return pd.read_csv(TRAIN_LOG_PATH).to_dict("records")
    return []


train_logs = load_existing_train_log()

for step in range(START_STEP + 1, TRAIN_STEPS + 1):
    row_index = int(train_schedule[step - 1])
    row = training_df.iloc[row_index]
    example = row_to_order_example(row_index, row)
    gold_order = example["order"]

    # Pass A: exact 24-class scores, preferably with one shared KV prefix.
    model.eval()
    reference_scores = score_full24_distribution(model, example)
    coefficients, loss_values, _ = score_space_gradient_coefficients(reference_scores, gold_order)
    distribution_log = training_distribution_log(reference_scores, gold_order)

    # Pass B: exact chain-rule gradient with tiny candidate chunks.
    optimizer.zero_grad(set_to_none=True)
    max_rescore_diff, rescored_tensor = backward_full24_exact(
        model,
        example,
        coefficients,
        reference_scores,
    )
    if max_rescore_diff > RESCORE_SCORE_TOLERANCE:
        optimizer.zero_grad(set_to_none=True)
        raise RuntimeError(
            f"Distribution/rescore mismatch too large at step {step}: "
            f"{max_rescore_diff:.6f} > {RESCORE_SCORE_TOLERANCE:.6f}. "
            "Disable KV or inspect dropout/token scoring before continuing."
        )

    grad_norm = float(torch.nn.utils.clip_grad_norm_(
        [parameter for parameter in model.parameters() if parameter.requires_grad],
        MAX_GRAD_NORM,
    ).detach().cpu().item())
    optimizer.step()

    top1_index = int(torch.argmax(reference_scores).item())
    log_row = {
        "step": step,
        "sample_id": str(row["Id"]),
        "gold_order": compact_order(gold_order),
        "pre_top1_order": compact_order(PERMUTATIONS[top1_index]),
        "distribution_scorer": "kv" if USE_KV_DISTRIBUTION_SCORER else "recompute",
        **loss_values,
        **distribution_log,
        "score_coefficient_l1": float(coefficients.abs().sum().item()),
        "score_coefficient_max_abs": float(coefficients.abs().max().item()),
        "rescore_max_abs_score_diff": max_rescore_diff,
        "grad_norm": grad_norm,
        "learning_rate": float(optimizer.param_groups[0]["lr"]),
    }
    train_logs.append(log_row)
    pd.DataFrame(train_logs).to_csv(TRAIN_LOG_PATH, index=False)

    if step % LOGGING_STEPS == 0 or step == START_STEP + 1:
        print(
            f"step={step:03d} "
            f"loss={loss_values['total_loss']:.4f} "
            f"gold_p={distribution_log['pre_gold_order_probability']:.4f} "
            f"first_p={distribution_log['pre_first_gold_probability']:.4f} "
            f"last_p={distribution_log['pre_last_gold_probability']:.4f} "
            f"full_Hn={distribution_log['pre_full_normalized_entropy']:.4f} "
            f"first_Hn={distribution_log['pre_first_normalized_entropy']:.4f} "
            f"last_Hn={distribution_log['pre_last_normalized_entropy']:.4f} "
            f"rescore_diff={max_rescore_diff:.6f}"
        )

    should_save = (step % SAVE_STEPS == 0) or (step == TRAIN_STEPS)
    should_eval = (step % EVAL_STEPS == 0) or (step == TRAIN_STEPS)

    if should_save:
        checkpoint_dir = save_training_checkpoint(step)

    if should_eval:
        evaluate_model(
            model,
            quick_eval_df,
            tag=f"step_{step:03d}_quick20",
            output_dir=QUICK_EVAL_DIR,
        )

    del reference_scores, coefficients, rescored_tensor
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

print("Training complete.")
print("Train log:", TRAIN_LOG_PATH)


In [ ]:

# 11) Final matched100 evaluation and checkpoint comparison
if RUN_FINAL_MATCHED100_EVAL:
    final_detail_df, final_summary_df = evaluate_model(
        model,
        full_eval_df,
        tag=f"step_{TRAIN_STEPS:03d}_matched100",
        output_dir=FULL_EVAL_DIR,
    )
else:
    print("Final matched100 evaluation disabled.")

summary_files = sorted(QUICK_EVAL_DIR.glob("*_summary.csv")) + sorted(FULL_EVAL_DIR.glob("*_summary.csv"))
summary_frames = []
for path in summary_files:
    frame = pd.read_csv(path)
    frame["summary_file"] = str(path)
    summary_frames.append(frame)
if summary_frames:
    comparison_df = pd.concat(summary_frames, ignore_index=True)
    comparison_path = EVAL_DIR / "full24_checkpoint_comparison.csv"
    comparison_df.to_csv(comparison_path, index=False)
    display(comparison_df[[
        "tag", "rows", "exact", "first", "last", "both_endpoints",
        "mean_gold_order_probability", "mean_gold_order_rank",
        "mean_order_brier", "mean_first_brier", "mean_last_brier",
        "full_entropy_error_auc", "first_entropy_error_auc", "last_entropy_error_auc",
        "pairwise_first", "pairwise_last", "pairwise_both_endpoints",
        "order_pairwise_first_disagreement_rate",
        "order_pairwise_last_disagreement_rate",
        "any_endpoint_disagreement_exact_error_auc",
        "exact_when_endpoint_disagree_count_0",
        "exact_when_endpoint_disagree_count_1",
        "exact_when_endpoint_disagree_count_2",
        "global_low_count", "global_low_exact",
        "global_low_first_entropy_error_auc", "global_low_last_entropy_error_auc",
        "global_low_any_endpoint_disagreement_exact_error_auc",
    ]])
    print("saved:", comparison_path)


## 결과 판정 기준

이 실험은 단순히 Exact만 보는 것이 아니라 다음을 함께 확인합니다.

### 정확도

- `exact`
- `first`, `last`
- `both_endpoints`
- `position`, `pair`

### 전체 순열 분포

- `gold_order_probability`, `gold_order_rank`, `gold_order_nll`
- `full_top1_margin`
- `full_normalized_entropy`
- `order_brier`
- `full_entropy_error_auc`

### 첫·마지막 주변분포

- `first/last_gold_probability`
- `first/last_marginal_margin`
- `first/last_normalized_entropy`
- `first/last_brier`
- `first/last_entropy_error_auc`

### Pairwise 및 합의 지표

Pairwise 확률은 별도 추론 없이 full-24 확률에서 계산합니다.

- `p_pair_i_before_j`
- `pairwise_early_score_1..4`, `pairwise_late_score_1..4`
- `pairwise_first_prediction`, `pairwise_last_prediction`
- `pairwise_first_margin`, `pairwise_last_margin`
- `pairwise_cycle_count`, `pairwise_transitivity_score`
- `order_pairwise_first_disagree`, `order_pairwise_last_disagree`
- `order_pairwise_endpoint_disagree_count`
- marginal / pairwise / conditional의 `threeway` 합의 여부

폴백 라우팅 신호로 유용한지는 다음을 우선 확인합니다.

```text
first_accuracy_when_order_pairwise_agree
first_accuracy_when_order_pairwise_disagree
last_accuracy_when_order_pairwise_agree
last_accuracy_when_order_pairwise_disagree

exact_when_endpoint_disagree_count_0
exact_when_endpoint_disagree_count_1
exact_when_endpoint_disagree_count_2

any_endpoint_disagreement_exact_error_auc
global_low_any_endpoint_disagreement_exact_error_auc
```

불일치 개수가 증가할수록 Exact가 뚜렷하게 낮아지고, disagreement 기반 error AUC가 0.5보다 충분히 높아질 때 폴백 위치 선택 신호로 사용할 근거가 생깁니다.

### 기존 위치별 confidence 문제

특히 기존 문제를 해결했는지는 다음 두 값으로 확인합니다.

```text
global_low_first_entropy_error_auc
global_low_last_entropy_error_auc
```

기존 quick20에서 이 값들은 약 0.4 수준으로 위치별 오류를 구분하지 못했습니다. 새 학습 후에는 Exact와 endpoint 정확도를 해치지 않으면서 이 값이 0.5를 넘고, 가능하면 지속적으로 상승하는지가 중요합니다.

`KV_VERIFIED=False`이면 노트북은 자동으로 독립 재계산 scorer를 사용합니다. 이 경우에도 loss와 gradient 정의는 동일하고 속도만 달라집니다.
